In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np

#To use MIMIC which we should not need once all data is in CLIF format
use_mimic = True

#file paths
path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"
path_clif =  "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mimicclif"
if use_mimic:
    path_mimic =  "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mimiciv"

#Borrowed from Jinping
def _ensure_datetime_naive(s: pd.Series) -> pd.Series:
    """Parse to datetime; if tz-aware, drop tz to make naive."""
    s = pd.to_datetime(s, errors='coerce')
    try:
        return s.dt.tz_localize(None)
    except TypeError:
        return s

def _ensure_datetime_est(s: pd.Series) -> pd.Series:
    """Parse to datetime; convert UTC into EST, drop tz to make naive."""
    s = pd.to_datetime(s, errors='coerce')
    try:
        # Convert to EST if timezone-aware
        if s.dt.tz is not None:
            s = s.dt.tz_convert('America/New_York')
        return s.dt.tz_localize(None)
    except (TypeError, AttributeError):
        return s

#Used to break variables into time windows
def classify_time_window(td):
    hours = td.total_seconds() / 3600
    if 0 <= hours < 24:
        return '0-24h'
    elif 24 <= hours < 48:
        return '24-48h'
    elif 48 <= hours < 72:
        return '48-72h'
    else:
        return None

def classify_time_window_prior(td):
    hours = td.total_seconds() / 3600
    if -24 <= hours < 0:
        return 'prior24h'
    elif 0 <= hours < 24:
        return '0-24h'
    elif 24 <= hours < 48:
        return '24-48h'
    elif 48 <= hours < 72:
        return '48-72h'
    elif hours >=72:
        return 'post72h'
    else:
        return None

block_df = pd.read_parquet(os.path.join(path_ptot,"PT_prior_post_imv.parquet"))
resp_df = pd.read_parquet(os.path.join(path_clif,"clif_respiratory_support.parquet"))
vitals_df = pd.read_parquet(os.path.join(path_clif,"clif_vitals.parquet"))
patient_df = pd.read_parquet(os.path.join(path_clif,"clif_patient.parquet"))
ass_df = pd.read_parquet(os.path.join(path_clif,"clif_patient_assessments.parquet"))
adt_df = pd.read_parquet(os.path.join(path_clif,"clif_adt.parquet"))
hosp_df = pd.read_parquet(os.path.join(path_clif,"clif_hospitalization.parquet"))

####TESTING ONLY####################################
#block_df = block_df.head(100) #smallercohort for development
#print("WARNING: Smaller cohort size for testing.")

#remove the ones not in cohort and other not useful columns
vitals_df = vitals_df[vitals_df['hospitalization_id'].isin( block_df['hospitalization_id'] )].reset_index()
vitals_df = vitals_df.drop(columns=['vital_name','meas_site_name'])
resp_df = resp_df[resp_df['hospitalization_id'].isin( block_df['hospitalization_id'] )].reset_index()
patient_df = patient_df[patient_df['patient_id'].isin ( block_df['patient_id'] )].reset_index()
patient_df = patient_df[['patient_id','sex_category','race_category','ethnicity_category','birth_date','language_category']]
ass_df = ass_df[ass_df['hospitalization_id'].isin( block_df['hospitalization_id'] )].reset_index()
ass_df = ass_df.drop(columns=['assessment_name','assessment_group','categorical_value','text_value'])
adt_df = adt_df[adt_df['hospitalization_id'].isin( block_df['hospitalization_id'] )].reset_index()
hosp_df = hosp_df[hosp_df['hospitalization_id'].isin(block_df['hospitalization_id'])].reset_index()

#fix timezone issues
#otpt_df has already some known to be TZ naive
resp_df['recorded_dttm'] = _ensure_datetime_est(resp_df['recorded_dttm'])
vitals_df['recorded_dttm'] = _ensure_datetime_est(vitals_df['recorded_dttm'])
block_df['discharge_dttm'] = _ensure_datetime_naive(block_df['discharge_dttm'])
block_df['block_first_vital_dttm'] = _ensure_datetime_naive(block_df['block_first_vital_dttm'])
block_df['block_last_vital_dttm'] = _ensure_datetime_naive(block_df['block_last_vital_dttm'])
block_df['death_dttm'] = _ensure_datetime_naive(block_df['death_dttm'])
block_df['final_outcome_dttm'] = _ensure_datetime_naive(block_df['final_outcome_dttm'])
ass_df['recorded_dttm'] = _ensure_datetime_est(ass_df['recorded_dttm'])
adt_df['out_dttm'] = _ensure_datetime_est(adt_df['out_dttm'])
adt_df['in_dttm'] = _ensure_datetime_est(adt_df['in_dttm'])
hosp_df['admission_dttm'] = _ensure_datetime_est(hosp_df['admission_dttm'])

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")
print(f"Unique Admission IDs: {block_df['hospitalization_id'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126
Unique Admission IDs: 32136


In [2]:
## FOR Sex, Race and Ethnicity, and Language

#merge
block_df = pd.merge(
    block_df,
    patient_df,
    on='patient_id',
    how='left'
)

#Convert birth date to date object
if not use_mimic:
    block_df['birth_date'] = pd.to_date(block_df['birth_date'], errors='coerce')
    block_df['age'] = block_df['block_vent_start_dttm'].dt.year - block_df['birth_date'].dt.year

#remove useless column
block_df = block_df.drop(columns=['birth_date'])

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [3]:
##AGE, LANGUAGE, Payor from MIMIC data given missing on CLIF'd tables.
"""
Language and Payor for MIMIC is extracted from the admissions table.
Our CLIF'd tables are missing birth date since this is not in MIMIC but rather uses anchor_year and anchor_age.
We are calculating age at the time of intubation.
Also, note that MIMIC age stops at 89, all patients older than 89 are assiged anchor_age 91.
Language for MIMIC is extracted from the admissions table.
"""
if use_mimic:
    #AGE
    
    #load
    patient_mimic_df = pd.read_parquet(os.path.join(path_mimic,"hosp","patients.parquet"))
    patient_mimic_df.rename(columns={'subject_id': 'patient_id'}, inplace=True)
    patient_mimic_df['patient_id'] = patient_mimic_df['patient_id'].astype(str)
    
    #filter
    patient_mimic_df = patient_mimic_df[patient_mimic_df['patient_id'].isin( block_df['patient_id'] )]
    print(f"Unique MIMIC patient id: {patient_mimic_df['patient_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['patient_id'].nunique()}")
    
    #merge
    merged_patient_df = pd.merge(
        block_df,
        patient_mimic_df,
        on='patient_id',
        how='left'
    )
    #convert admission time to year and calculate year - anchor_year + anchor_age
    merged_patient_df['age'] = merged_patient_df['block_vent_start_dttm'].dt.year - merged_patient_df['anchor_year'] + merged_patient_df['anchor_age']

    #Date of death
    merged_patient_df["date_of_death"] = pd.to_datetime(merged_patient_df["dod"])

    #Admission Year (based on estimate from anchor year
    merged_patient_df["anchor_year_group"] = merged_patient_df["anchor_year_group"].str.slice(0, 4).astype(int) + 1 #Convert achor year group to an integer
    merged_patient_df["admission_year"] = block_df['block_first_vital_dttm'].dt.year.astype(int) - merged_patient_df["anchor_year"] + merged_patient_df["anchor_year_group"]
    
    #drop unnecessary columns
    block_df = merged_patient_df.drop(columns=['gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'])
    
    ###LANGUAGE & PAYOR
    
    #For Language, we need to get this from a different table.
    block_df = block_df.drop(columns=['language_category'])
    
    #load
    adm_mimic_df = pd.read_parquet(os.path.join(path_mimic,"hosp","admissions.parquet"))
    adm_mimic_df.rename(columns={'hadm_id': 'hospitalization_id'}, inplace=True)
    adm_mimic_df['hospitalization_id'] = adm_mimic_df['hospitalization_id'].astype(str)
    adm_mimic_df = adm_mimic_df[['hospitalization_id','language','insurance']]
    adm_mimic_df['insurance'] = adm_mimic_df['insurance'].astype(str)
    adm_mimic_df['language'] = adm_mimic_df['language'].astype(str)
    
    #filter
    adm_mimic_df = adm_mimic_df[adm_mimic_df['hospitalization_id'].isin( block_df['hospitalization_id'] )]
    print(f"Unique MIMIC hosp id: {adm_mimic_df['hospitalization_id'].nunique()}")
    print(f"Unique Block patient id: {block_df['hospitalization_id'].nunique()}")
    
    #merge
    block_df = pd.merge(
        block_df,
        adm_mimic_df,
        on='hospitalization_id',
        how='left'
    )

    del patient_mimic_df, merged_patient_df, adm_mimic_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Unique MIMIC patient id: 29396
Unique Block patient id: 29396
Unique MIMIC hosp id: 32136
Unique Block patient id: 32136
Block Length: 32276
Unique Encounter Block: 32126


In [4]:
block_df.columns

Index(['patient_id', 'hospitalization_id', 'encounter_block',
       'discharge_category', 'discharge_dttm', 'block_vent_start_dttm',
       'block_vent_end_dttm', 'block_first_vital_dttm',
       'block_last_vital_dttm', 'death_dttm', 'final_outcome_dttm', 'is_dead',
       'first_ot_post_imv_dttm', 'first_pt_post_imv_dttm',
       'first_any_post_imv_dttm', 'Time_first_OT', 'Time_first_PT',
       'Time_first_OT_or_PT', 'yellow_time_eligibility', 'delayed_yellow_PT',
       'delayed_yellow_OT', 'delayed_yellow_OT_or_PT', 'last_pt_pre_imv_dttm',
       'Time_last_PT', 'sex_category', 'race_category', 'ethnicity_category',
       'age', 'date_of_death', 'admission_year', 'language', 'insurance'],
      dtype='object')

In [5]:
##HEART RATE

#Filter vitals_df for heart_rate only and merge
hr_df = vitals_df[vitals_df['vital_category'] == 'heart_rate'].copy()
merged_hr_df = pd.merge(
    block_df,
    hr_df,
    on='hospitalization_id',
    how='inner'
)

#Calculate time windows from block_vent_start_dttm
merged_hr_df['time_diff'] = merged_hr_df['recorded_dttm'] - merged_hr_df['block_vent_start_dttm']
merged_hr_df['time_window'] = merged_hr_df['time_diff'].apply(classify_time_window)
merged_hr_df = merged_hr_df[merged_hr_df['time_window'].notnull()]

#Group by encounter block and by time window, calculate mean
grouped_hr_df = (
    merged_hr_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_hr_df.rename(columns={
    '0-24h': 'mean_heart_rate_0_24h',
    '24-48h': 'mean_heart_rate_24_48h',
    '48-72h': 'mean_heart_rate_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_hr_df,
    on='encounter_block',
    how='left'
)

del hr_df, merged_hr_df, grouped_hr_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [6]:
##MAP

#First create MAP out of SBP and DBP when MAP is missing.

#Filter vitals_df to only relevant categories
bp_df = vitals_df[
    vitals_df['vital_category'].isin(['sbp', 'dbp', 'map'])
].copy()

#Pivot to get sbp, dbp, and map in columns per hospitalization_id + recorded_dttm
pivoted_bp_df = bp_df.pivot_table(
    index=['hospitalization_id', 'recorded_dttm'],
    columns='vital_category',
    values='vital_value',
    aggfunc='first'  # in case there are multiple values per timestamp, take the first
).reset_index()

#Identify rows where MAP is missing but SBP and DBP are present
mask_missing_map = (
    pivoted_bp_df['map'].isna() &
    pivoted_bp_df['sbp'].notna() &
    pivoted_bp_df['dbp'].notna()
)

#Compute synthetic MAP
synthetic_map_rows = pivoted_bp_df[mask_missing_map].copy()
synthetic_map_rows['vital_value'] = (
    synthetic_map_rows['sbp'] + 2 * synthetic_map_rows['dbp']
) / 3
synthetic_map_rows['vital_category'] = 'map'

#Keep only necessary columns for synthetic rows
synthetic_map_rows = synthetic_map_rows[[
    'hospitalization_id', 'recorded_dttm', 'vital_category', 'vital_value'
]]

#Get existing 'map' rows (real ones)
real_map_rows = bp_df[
    bp_df['vital_category'] == 'map'
][[
    'hospitalization_id', 'recorded_dttm', 'vital_category', 'vital_value'
]]

#Combine real and synthetic MAP rows
map_df = pd.concat([real_map_rows, synthetic_map_rows], ignore_index=True)


#merge
merged_map_df = pd.merge(
    block_df,
    map_df,
    on='hospitalization_id',
    how='inner'
)

#Calculate time windows from block_vent_start_dttm
merged_map_df['time_diff'] = merged_map_df['recorded_dttm'] - merged_map_df['block_vent_start_dttm']
merged_map_df['time_window'] = merged_map_df['time_diff'].apply(classify_time_window)
merged_map_df = merged_map_df[merged_map_df['time_window'].notnull()]

#Group by encounter block and by time window, calculate mean
grouped_map_df = (
    merged_map_df.groupby(['encounter_block', 'time_window'])['vital_value']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_map_df.rename(columns={
    '0-24h': 'mean_map_0_24h',
    '24-48h': 'mean_map_24_48h',
    '48-72h': 'mean_map_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_map_df,
    on='encounter_block',
    how='left'
)

del bp_df, pivoted_bp_df, synthetic_map_rows, real_map_rows, map_df, merged_map_df, grouped_map_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [7]:
#BMI

#Filter vitals table for height and weight only
bmi_df = vitals_df[
    vitals_df['vital_category'].isin(['height_cm', 'weight_kg'])
].copy()

# Remove outliers
# For height rows: set out-of-range to NaN
is_height = bmi_df['vital_category'] == 'height_cm'
height_mask_low  = is_height & (bmi_df['vital_value'] < 76)
height_mask_high = is_height & (bmi_df['vital_value'] > 255)
bmi_df.loc[height_mask_low | height_mask_high, 'vital_value'] = np.nan

# For weight rows: set out-of-range to NaN
is_weight = bmi_df['vital_category'] == 'weight_kg'
weight_mask_low  = is_weight & (bmi_df['vital_value'] < 20)
weight_mask_high = is_weight & (bmi_df['vital_value'] > 1100)
bmi_df.loc[weight_mask_low | weight_mask_high, 'vital_value'] = np.nan

# Merge with vent_start_end to get ventilation start time
merged_bmi_df = pd.merge(
    block_df[['encounter_block','hospitalization_id','block_vent_start_dttm']],
    bmi_df,
    on='hospitalization_id',
    how='inner'
)

# Calculate time difference between recorded_dttm and vent_start_time
merged_bmi_df['time_diff'] = (merged_bmi_df['recorded_dttm'] - merged_bmi_df['block_vent_start_dttm']).dt.total_seconds()

# Define whether measurement is before or after vent_start_time
merged_bmi_df['before_vent_start'] = (merged_bmi_df['time_diff'] <= 0).astype(int)

# Calculate absolute time difference
merged_bmi_df['abs_time_diff'] = merged_bmi_df['time_diff'].abs()

# Sort data to prioritize measurements before vent start and closest in time
merged_bmi_df = merged_bmi_df.sort_values(['encounter_block', 'vital_category', 'before_vent_start', 'abs_time_diff'], 
                                    ascending=[True, True, False, True])

# Drop duplicates to keep the closest measurement for each vital_category per encounter block
merged_bmi_df = merged_bmi_df.drop_duplicates(subset=['encounter_block', 'vital_category'], keep='first')

# Pivot to get height and weight per encounter block
pivot_bmi_df = merged_bmi_df.pivot(index='encounter_block', 
                                    columns='vital_category', 
                                    values='vital_value'
                                    ).reset_index()

# Calculate BMI
pivot_bmi_df['bmi'] = pivot_bmi_df['weight_kg'] / ((pivot_bmi_df['height_cm'] / 100) ** 2)

#Add back to block
block_df = pd.merge(
    block_df,
    pivot_bmi_df,
    on='encounter_block',
    how='left'
)

del bmi_df, merged_bmi_df, pivot_bmi_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [8]:
##Respiratory
#Currently just FiO2 and PEEP set
fio2_df = resp_df.copy()

#Set all values to be from 0 to 1
def normalize_fio2(x):
    if 20 < x < 100:
        return x/100
    elif 0.2 < x < 1:
        return x
    else:
        return np.nan
fio2_df['fio2_set'] = resp_df['fio2_set'].apply(normalize_fio2)

#Fill in missing FiO2 for Other Modes
def synthetic_fio2(given, lpm, mode):
    #Delivery mode is more important than set FiO2
    #Room air is 21%
    if mode == "Room Air":
        return 0.21
    #NC is 20% + 4%xFlow if flow is given and within range. Otherwise it is NA.
    elif mode == "Nasal Cannula":
        if 0.5 < lpm <= 6:
            return 0.2 + 0.04*lpm
        else:
            return np.nan
    #Face mask, will have to assume simple since they do not mention NRB. Set to 5% + 5% x Flow but only if flow is within reasonable range. Else return
    elif mode == "Face Mask":
        if given:
            return given
        if 5 <= lpm <= 10:
            return 0.05 + 0.05*lpm
        else:
            return np.nan
    #Trach collar will assume room air if no specific FiO2 is given.
    elif mode == "Trach Collar":
        if given:
            return given
        else:
            return 0.21
    else: #This would include IMV and NIPPV.
        return given
fio2_df['fio2_set'] = fio2_df.apply(lambda row: synthetic_fio2(row['fio2_set'],row['lpm_set'],row['device_category']), axis=1)

#Remove missing data and keep only relevant columns
#fio2_df = fio2_df[fio2_df['fio2_set'].notna() or fio2_df['peep_set'].notna()]
fio2_df = fio2_df[['hospitalization_id','recorded_dttm','fio2_set','peep_set']]

merged_resp_df = pd.merge(
    block_df,
    fio2_df,
    on='hospitalization_id',
    how='inner'
)

#Calculate time windows from block_vent_start_dttm
merged_resp_df['time_diff'] = merged_resp_df['recorded_dttm'] - merged_resp_df['block_vent_start_dttm']
merged_resp_df['time_window'] = merged_resp_df['time_diff'].apply(classify_time_window)
merged_resp_df = merged_resp_df[merged_resp_df['time_window'].notnull()]

#Group by encounter block and by time window, calculate mean for Fio2
grouped_fio2_df = (
    merged_resp_df.groupby(['encounter_block', 'time_window'])['fio2_set']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)
#Rename columns for clarity
grouped_fio2_df.rename(columns={
    '0-24h': 'mean_fio2_0_24h',
    '24-48h': 'mean_fio2_24_48h',
    '48-72h': 'mean_fio2_48_72h'
}, inplace=True)
block_df = pd.merge(
    block_df,
    grouped_fio2_df,
    on='encounter_block',
    how='left'
)


#Group by encounter block and by time window, calculate mean for PEEP
grouped_peep_df = (
    merged_resp_df.groupby(['encounter_block', 'time_window'])['peep_set']
    .mean()
    .unstack(fill_value=None)
    .reset_index()
)
#Rename columns for clarity
grouped_peep_df.rename(columns={
    '0-24h': 'mean_peep_0_24h',
    '24-48h': 'mean_peep_24_48h',
    '48-72h': 'mean_peep_48_72h'
}, inplace=True)
block_df = pd.merge(
    block_df,
    grouped_peep_df,
    on='encounter_block',
    how='left'
)

del fio2_df, merged_resp_df, grouped_fio2_df, grouped_peep_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [9]:
#Intubation status at the time of order
#Currently using the either PT ot OT time, no specific to one type of order.
block_df['imv_at_order'] = block_df['block_vent_end_dttm'] > block_df['first_any_post_imv_dttm']

In [10]:
#SOFA score
#The mobilization code already calculated for the first 24 hours. Other data could be extratected if desired. There is a dedicated python script to do this in their code.
sofa_df = pd.read_parquet(os.path.join(path_mob,"intermediate","sofa.parquet"))
sofa_df = sofa_df[['encounter_block','sofa_total']]
sofa_df.rename(columns={'sofa_total':'sofa_24h'})
block_df = pd.merge(
    block_df,
    sofa_df,
    on='encounter_block',
    how='left'
)
del sofa_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [11]:
#Braden Mobility Score

#Filter ass_df for braden_mob only and merge
braden_df = ass_df[ass_df['assessment_category'] == 'braden_mobility'].copy()
merged_braden_df = pd.merge(
    block_df,
    braden_df,
    on='hospitalization_id',
    how='inner'
)

#Calculate time windows from block_vent_start_dttm
merged_braden_df['time_diff'] = merged_braden_df['recorded_dttm'] - merged_braden_df['block_vent_start_dttm']
merged_braden_df['time_window'] = merged_braden_df['time_diff'].apply(classify_time_window_prior)
merged_braden_df = merged_braden_df[merged_braden_df['time_window'].notnull()]

#Group by encounter block and by time window, calculate mean
grouped_braden_df = (
    merged_braden_df.groupby(['encounter_block', 'time_window'])['numerical_value']
    .max()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_braden_df.rename(columns={
    'prior24h':'max_braden_prior24h',
    '0-24h': 'max_braden_0_24h',
    '24-48h': 'max_braden_24_48h',
    '48-72h': 'max_braden_48_72h',
    'post72h': 'max_braden_post72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_braden_df,
    on='encounter_block',
    how='left'
)

del braden_df, merged_braden_df, grouped_braden_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [12]:
#RASS

#Filter ass_df for RASS only and merge
rass_df = ass_df[ass_df['assessment_category'] == 'RASS'].copy()
merged_rass_df = pd.merge(
    block_df,
    rass_df,
    on='hospitalization_id',
    how='inner'
)

#Calculate time windows from block_vent_start_dttm
merged_rass_df['time_diff'] = merged_rass_df['recorded_dttm'] - merged_rass_df['block_vent_start_dttm']
merged_rass_df['time_window'] = merged_rass_df['time_diff'].apply(classify_time_window)
merged_rass_df = merged_rass_df[merged_rass_df['time_window'].notnull()]

#Group by encounter block and by time window, calculate mean
grouped_rass_df = (
    merged_rass_df.groupby(['encounter_block', 'time_window'])['numerical_value']
    .min()
    .unstack(fill_value=None)
    .reset_index()
)

#Rename columns for clarity
grouped_rass_df.rename(columns={
    '0-24h': 'min_RASS_0_24h',
    '24-48h': 'min_RASS_24_48h',
    '48-72h': 'min_RASS_48_72h'
}, inplace=True)

block_df = pd.merge(
    block_df,
    grouped_rass_df,
    on='encounter_block',
    how='left'
)

del rass_df, merged_rass_df, grouped_rass_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [13]:
#Paralytics and pressors

#Get the hourly data from the mobilization output
hourly_df = pd.read_parquet(os.path.join(path_mob,"intermediate","final_df_hourly.parquet"))
hourly_df = hourly_df[['encounter_block','time_from_vent','ne_calc_max','paralytics_flag']]

#filter out the values outside the time window of interest, first 24 hours
hourly_df = hourly_df[(hourly_df['time_from_vent'] >= 0) & (hourly_df['time_from_vent'] < 24)]

#Create paralytics data frame
hour_paralytics_df = (
    hourly_df.groupby(['encounter_block'])['paralytics_flag']
    .sum()
    .reset_index()
)
hour_paralytics_df['paralytics_24h'] = hour_paralytics_df['paralytics_flag'] >= 4
#merge back to blocks
block_df = pd.merge(
    block_df,
    hour_paralytics_df[['encounter_block','paralytics_24h']],
    on='encounter_block',
    how='left'
)

#Create pressors data frame
pressor_df = (
    hourly_df.groupby(['encounter_block'])['ne_calc_max']
    .max()
    .reset_index()
)
pressor_df['pressor_24h'] = pressor_df['ne_calc_max'] > 0
#merge back to blocks
block_df = pd.merge(
    block_df,
    pressor_df[['encounter_block','pressor_24h']],
    on='encounter_block',
    how='left'
)

del hourly_df, hour_paralytics_df, pressor_df

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")

Block Length: 32276
Unique Encounter Block: 32126


In [14]:
#ICU ADT data

merged_adt_df = pd.merge (
    block_df[['encounter_block','hospitalization_id','block_vent_start_dttm']],
    adt_df,
    on='hospitalization_id',
    how = 'inner'
)
#keep ICUs only
merged_adt_df = merged_adt_df[merged_adt_df['location_category']== 'icu']

#Sort by in_dttm
merged_adt_df= merged_adt_df.sort_values(['encounter_block','in_dttm'], ascending=True)

#ICU Type
#remove anywhere the patient left before start of IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']]
ICU_df = ICU_df.drop_duplicates(subset=['encounter_block'], keep='first')
ICU_df.rename(columns={'location_name':'ICU_type'}, inplace=True)
ICU_df = ICU_df[['encounter_block','ICU_type']]
#Merge back to block
block_df = block_df.merge(
    ICU_df,
    on='encounter_block',
    how='left'
)
block_df['ICU_type'] = block_df['ICU_type'].astype(str)


#ICU Re-admission, post IMV
ICU_df = merged_adt_df[merged_adt_df['out_dttm'] > merged_adt_df['block_vent_start_dttm']]
ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.
icu_readmit_df = (
   ICU_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_readmission=("icu_readmission", "max"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_readmit_df[["encounter_block", "icu_readmission"]],
    on=["encounter_block"],
    how="left"
)
block_df["icu_readmission"] = block_df["icu_readmission"].astype(bool)

#ICU stays in total
#Whole encounter block, not just post-IMV
icu_N_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_N=("in_dttm", "count"))
)
#Marge back to blocks
block_df = block_df.merge(
    icu_N_df[["encounter_block", "icu_N"]],
    on=["encounter_block"],
    how="left"
)

#ICU LOS, including whole encounter block, not just post IMV
merged_adt_df["icu_los_days"] = (merged_adt_df["out_dttm"] - merged_adt_df["in_dttm"]).dt.total_seconds() / (24*3600)
icu_los_df = (
    merged_adt_df.groupby(["encounter_block"], as_index=False)
    .agg(icu_los_days=("icu_los_days", "sum"))
)
#Merge back to blocks
block_df = block_df.merge(
    icu_los_df[["encounter_block", "icu_los_days"]],
    on=["encounter_block"],
    how="left"
)

del merged_adt_df, ICU_df
print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}")


Block Length: 32276
Unique Encounter Block: 32126


/tmp/ipykernel_1414655/3579218917.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ICU_df['icu_readmission'] = ICU_df['encounter_block'].duplicated(keep='first') #If an encounter block is duplicated after removing pre-IMV ICU stays, this implies re-admission.


In [15]:
#CHARLSON COMORBIDITY INDEX

if use_mimic:
    diag_mimic_df = pd.read_csv(os.path.join(path_mimic,"hosp","diagnoses_icd.csv.gz"))
    diag_mimic_df.rename(columns={'hadm_id':'hospitalization_id'}, inplace=True)
    diag_mimic_df['hospitalization_id'] = diag_mimic_df['hospitalization_id'].astype(str)

    #filter for only relevant hospitalizations
    diag_mimic_df = diag_mimic_df[diag_mimic_df['hospitalization_id'].isin( block_df['hospitalization_id'] )]

    #merge with encounter blocks
    diag_df = pd.merge(
        block_df[['encounter_block','hospitalization_id']],
        diag_mimic_df,
        on='hospitalization_id',
        how='inner'
    )

    del diag_mimic_df

    #The following code is taken from a SQL code from the mimic library and translated to Pandas form using ChatGPT
    diag_df["icd9_code"] = np.where(diag_df["icd_version"] == 9, diag_df["icd_code"], None)
    diag_df["icd10_code"] = np.where(diag_df["icd_version"] == 10, diag_df["icd_code"], None)
    
    def get_flags(g):
        """Compute Charlson comorbidity flags for one encounter group."""
        f = {}
    
        # Helper shortcuts
        icd9 = g["icd9_code"].fillna("")
        icd10 = g["icd10_code"].fillna("")
    
        # Myocardial infarction
        f["myocardial_infarct"] = int(any(
            icd9.str[:3].isin(["410","412"]) |
            icd10.str[:3].isin(["I21","I22"]) |
            (icd10.str[:4] == "I252")
        ))
    
        # Congestive heart failure
        f["congestive_heart_failure"] = int(any(
            (icd9.str[:3] == "428") |
            icd9.str[:5].isin([
                "39891","40201","40211","40291","40401","40403",
                "40411","40413","40491","40493"
            ]) |
            icd9.str[:4].between("4254","4259") |
            icd10.str[:3].isin(["I43","I50"]) |
            icd10.str[:4].isin([
                "I099","I110","I130","I132","I255","I420",
                "I425","I426","I427","I428","I429","P290"
            ])
        ))
    
        # Peripheral vascular disease
        f["peripheral_vascular_disease"] = int(any(
            icd9.str[:3].isin(["440","441"]) |
            icd9.str[:4].isin(["0930","4373","4471","5571","5579","V434"]) |
            icd9.str[:4].between("4431","4439") |
            icd10.str[:3].isin(["I70","I71"]) |
            icd10.str[:4].isin([
                "I731","I738","I739","I771","I790","I792",
                "K551","K558","K559","Z958","Z959"
            ])
        ))
    
        # Cerebrovascular disease
        f["cerebrovascular_disease"] = int(any(
            icd9.str[:3].between("430","438") |
            (icd9.str[:5] == "36234") |
            icd10.str[:3].isin(["G45","G46"]) |
            icd10.str[:3].between("I60","I69") |
            (icd10.str[:4] == "H340")
        ))
    
        # Dementia
        f["dementia"] = int(any(
            (icd9.str[:3] == "290") |
            icd9.str[:4].isin(["2941","3312"]) |
            icd10.str[:3].isin(["F00","F01","F02","F03","G30"]) |
            icd10.str[:4].isin(["F051","G311"])
        ))
    
        # Chronic pulmonary disease
        f["chronic_pulmonary_disease"] = int(any(
            icd9.str[:3].between("490","505") |
            icd9.str[:4].isin(["4168","4169","5064","5081","5088"]) |
            icd10.str[:3].between("J40","J47") |
            icd10.str[:3].between("J60","J67") |
            icd10.str[:4].isin(["I278","I279","J684","J701","J703"])
        ))
    
        # Rheumatic disease
        f["rheumatic_disease"] = int(any(
            (icd9.str[:3] == "725") |
            icd9.str[:4].isin(["4465","7100","7101","7102","7103",
                               "7104","7140","7141","7142","7148"]) |
            icd10.str[:3].isin(["M05","M06","M32","M33","M34"]) |
            icd10.str[:4].isin(["M315","M351","M353","M360"])
        ))
    
        # Peptic ulcer disease
        f["peptic_ulcer_disease"] = int(any(
            icd9.str[:3].isin(["531","532","533","534"]) |
            icd10.str[:3].isin(["K25","K26","K27","K28"])
        ))
    
        # Mild liver disease
        f["mild_liver_disease"] = int(any(
            icd9.str[:3].isin(["570","571"]) |
            icd9.str[:4].isin(["0706","0709","5733","5734","5738","5739","V427"]) |
            icd9.str[:5].isin(["07022","07023","07032","07033","07044","07054"]) |
            icd10.str[:3].isin(["B18","K73","K74"]) |
            icd10.str[:4].isin([
                "K700","K701","K702","K703","K709","K713","K714","K715",
                "K717","K760","K762","K763","K764","K768","K769","Z944"
            ])
        ))
    
        # Diabetes without chronic complication
        f["diabetes_without_cc"] = int(any(
            icd9.str[:4].isin(["2500","2501","2502","2503","2508","2509"]) |
            icd10.str[:4].isin([
                "E100","E101","E106","E108","E109","E110","E111","E116","E118","E119",
                "E120","E121","E126","E128","E129","E130","E131","E136","E138","E139",
                "E140","E141","E146","E148","E149"
            ])
        ))
    
        # Diabetes with chronic complication
        f["diabetes_with_cc"] = int(any(
            icd9.str[:4].isin(["2504","2505","2506","2507"]) |
            icd10.str[:4].isin([
                "E102","E103","E104","E105","E107","E112","E113","E114","E115","E117",
                "E122","E123","E124","E125","E127","E132","E133","E134","E135","E137",
                "E142","E143","E144","E145","E147"
            ])
        ))
    
        # Hemiplegia / Paraplegia
        f["paraplegia"] = int(any(
            icd9.str[:3].isin(["342","343"]) |
            icd9.str[:4].isin(["3341","3440","3441","3442","3443","3444","3445","3446","3449"]) |
            icd10.str[:3].isin(["G81","G82"]) |
            icd10.str[:4].isin(["G041","G114","G801","G802","G830","G831","G832","G833","G834","G839"])
        ))
    
        # Renal disease
        f["renal_disease"] = int(any(
            icd9.str[:3].isin(["582","585","586","V56"]) |
            icd9.str[:4].isin(["5880","V420","V451"]) |
            icd9.str[:4].between("5830","5837") |
            icd9.str[:5].isin(["40301","40311","40391","40402","40403","40412","40413","40492","40493"]) |
            icd10.str[:3].isin(["N18","N19"]) |
            icd10.str[:4].isin([
                "I120","I131","N032","N033","N034","N035","N036","N037",
                "N052","N053","N054","N055","N056","N057","N250",
                "Z490","Z491","Z492","Z940","Z992"
            ])
        ))
    
        # Malignant cancer
        f["malignant_cancer"] = int(any(
            icd9.str[:3].between("140","172") |
            icd9.str[:4].between("1740","1958") |
            icd9.str[:3].between("200","208") |
            (icd9.str[:4] == "2386") |
            icd10.str[:3].isin(["C43","C88"]) |
            icd10.str[:3].between("C00","C26") |
            icd10.str[:3].between("C30","C34") |
            icd10.str[:3].between("C37","C41") |
            icd10.str[:3].between("C45","C58") |
            icd10.str[:3].between("C60","C76") |
            icd10.str[:3].between("C81","C85") |
            icd10.str[:3].between("C90","C97")
        ))
    
        # Severe liver disease
        f["severe_liver_disease"] = int(any(
            icd9.str[:4].isin(["4560","4561","4562"]) |
            icd9.str[:4].between("5722","5728") |
            icd10.str[:4].isin(["I850","I859","I864","I982","K704","K711",
                                "K721","K729","K765","K766","K767"])
        ))
    
        # Metastatic solid tumor
        f["metastatic_solid_tumor"] = int(any(
            icd9.str[:3].isin(["196","197","198","199"]) |
            icd10.str[:3].isin(["C77","C78","C79","C80"])
        ))
    
        # AIDS/HIV
        f["aids"] = int(any(
            icd9.str[:3].isin(["042","043","044"]) |
            icd10.str[:3].isin(["B20","B21","B22","B24"])
        ))
    
        return pd.Series(f)
    
    # Apply grouping
    comorbidity_flags = (
        diag_df.groupby("encounter_block", group_keys=False)
               .apply(get_flags, include_groups=False)
               .reset_index()
    )
    
    # --- Charlson Comorbidity Index (no age adjustment here, since no age column in diag_df) ---
    comorbidity_flags["charlson_comorbidity_index"] = (
          comorbidity_flags["myocardial_infarct"]
        + comorbidity_flags["congestive_heart_failure"]
        + comorbidity_flags["peripheral_vascular_disease"]
        + comorbidity_flags["cerebrovascular_disease"]
        + comorbidity_flags["dementia"]
        + comorbidity_flags["chronic_pulmonary_disease"]
        + comorbidity_flags["rheumatic_disease"]
        + comorbidity_flags["peptic_ulcer_disease"]
        + np.maximum(comorbidity_flags["mild_liver_disease"], 3*comorbidity_flags["severe_liver_disease"])
        + np.maximum(2*comorbidity_flags["diabetes_with_cc"], comorbidity_flags["diabetes_without_cc"])
        + np.maximum(2*comorbidity_flags["malignant_cancer"], 6*comorbidity_flags["metastatic_solid_tumor"])
        + 2*comorbidity_flags["paraplegia"]
        + 2*comorbidity_flags["renal_disease"]
        + 6*comorbidity_flags["aids"]
    )
    
    block_df = pd.merge(
        block_df,
        comorbidity_flags[['encounter_block','charlson_comorbidity_index']],
        on='encounter_block',
        how='left'
    )

    del diag_df, comorbidity_flags

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}") 

Block Length: 32276
Unique Encounter Block: 32126


In [16]:
# Add admit_dttm and admit_type_name from hospitalization table
hosp_df = hosp_df[["patient_id", "hospitalization_id", "admission_dttm", "admission_type_name"]]

#Merge back to block_df
block_df = pd.merge(
    block_df,
    hosp_df,
    on=["patient_id", "hospitalization_id"],
    how='left'
)

block_df.rename(columns={"admission_type_name":"admission_source"}, inplace=True)

print(f"Block Length: {len(block_df)}")
print(f"Unique Encounter Block: {block_df['encounter_block'].nunique()}") 

Block Length: 32276
Unique Encounter Block: 32126


In [17]:
block_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32276 entries, 0 to 32275
Data columns (total 66 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   patient_id                  32276 non-null  object        
 1   hospitalization_id          32276 non-null  object        
 2   encounter_block             32276 non-null  int64         
 3   discharge_category          32276 non-null  object        
 4   discharge_dttm              32276 non-null  datetime64[us]
 5   block_vent_start_dttm       32276 non-null  datetime64[ns]
 6   block_vent_end_dttm         32276 non-null  datetime64[ns]
 7   block_first_vital_dttm      32276 non-null  datetime64[us]
 8   block_last_vital_dttm       32276 non-null  datetime64[us]
 9   death_dttm                  7922 non-null   datetime64[ns]
 10  final_outcome_dttm          32276 non-null  datetime64[us]
 11  is_dead                     32276 non-null  int64     

In [18]:
#Save it to a parquet file
block_df.to_parquet('block_compiled_no_hourly_all_hours.parquet')

In [1]:
#Add Hourly Block by Block Data
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np


#file paths
path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"
path_clif =  "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mimicclif"

#Borrowed from Jinping
def _ensure_datetime_naive(s: pd.Series) -> pd.Series:
    """Parse to datetime; if tz-aware, drop tz to make naive."""
    s = pd.to_datetime(s, errors='coerce')
    try:
        return s.dt.tz_localize(None)
    except TypeError:
        return s

def _ensure_datetime_est(s: pd.Series) -> pd.Series:
    """Parse to datetime; convert UTC into EST, drop tz to make naive."""
    s = pd.to_datetime(s, errors='coerce')
    try:
        # Convert to EST if timezone-aware
        if s.dt.tz is not None:
            s = s.dt.tz_convert('America/New_York')
        return s.dt.tz_localize(None)
    except (TypeError, AttributeError):
        return s

#Used to break variables into time windows
def classify_time_window(hours):
    if 0 <= hours < 24:
        return '0-24h'
    elif 24 <= hours < 48:
        return '24-48h'
    elif 48 <= hours < 72:
        return '48-72h'
    else:
        return None

def classify_time_window_prior(td):
    hours = td.total_seconds() / 3600
    if -24 <= hours < 0:
        return 'prior24h'
    elif 0 <= hours < 24:
        return '0-24h'
    elif 24 <= hours < 48:
        return '24-48h'
    elif 48 <= hours < 72:
        return '48-72h'
    else:
        return None

#Load up the hourly block final data
hourly_df = pd.read_parquet("/gpfs/gibbs/project/jain_snigdha/jl4286/CLIF-eligibility-for-mobilization/output/intermediate/final_df_w_criteria.parquet")
block_final = pd.read_parquet('block_compiled_no_hourly_all_hours.parquet')

#Add hospitalization ID column
hourly_df = hourly_df.merge(block_final[["hospitalization_id", "encounter_block"]], on="encounter_block", how="left")

#Load up RASS remove the ones not in cohort and other not useful columns
rass_df = pd.read_parquet(os.path.join(path_clif,"clif_patient_assessments.parquet"))
rass_df = rass_df[rass_df['assessment_category'] == 'RASS']
rass_df = rass_df[rass_df['hospitalization_id'].isin( block_final['hospitalization_id'] )].reset_index()
rass_df = rass_df[['hospitalization_id','recorded_dttm','numerical_value']]
rass_df = rass_df.rename(columns={'numerical_value':'RASS'})
#fix timezone issues
rass_df['recorded_dttm'] = _ensure_datetime_est(rass_df['recorded_dttm'])

#Add date and time column
rass_df['recorded_date'] = rass_df['recorded_dttm'].dt.date
rass_df['recorded_hour'] = rass_df['recorded_dttm'].dt.hour
rass_df = rass_df[['hospitalization_id', 'recorded_date', 'recorded_hour', 'RASS']].copy()
hourly_df['recorded_date'] = pd.to_datetime(hourly_df['recorded_date'])
rass_df['recorded_date'] = pd.to_datetime(rass_df['recorded_date'])
hourly_df['recorded_hour'] = hourly_df['recorded_hour'].astype(int)
rass_df['recorded_hour'] = rass_df['recorded_hour'].astype(int)

#Add RASS data to hourly blocks
hourly_df = hourly_df.merge(
    rass_df,
    on=['hospitalization_id', 'recorded_date', 'recorded_hour'],
    how='left')
hourly_df = hourly_df.sort_values(['encounter_block', 'recorded_date', 'recorded_hour'])
del rass_df

#Forward fill RASS
hourly_df['RASS'] = (
    hourly_df
    .groupby(['encounter_block'])['RASS']
    .transform(lambda x: x.ffill(limit=12))
)

#Add labels to divide into the chunks we want
hourly_df['time_from_vent'] = hourly_df['time_from_vent'].astype(int)
hourly_df['time_window'] = hourly_df['time_from_vent'].apply(classify_time_window)

#Define coma
hourly_df['coma'] = hourly_df['RASS'] <= -2
hourly_df['coma'] = hourly_df['coma'].astype(int)

#Group by encounter block and by time window, calculate total time in coma
grouped_coma_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['coma']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_coma_df.rename(columns={
    '0-24h': 'coma_0_24h',
    '24-48h': 'coma_24_48h',
    '48-72h': 'coma_48_72h'
}, inplace=True)

block_final = pd.merge(
    block_final,
    grouped_coma_df,
    on='encounter_block',
    how='left'
)

del grouped_coma_df


## Group by encounter block and by time window, calculate total time in yellow eligibility all hours
#hourly_df['yellow_not_all_green'] = hourly_df['any_yellow_or_green_no_red_all_hours'].astype(int)
grouped_yellow_df = (
    hourly_df.groupby(['encounter_block', 'time_window'])['any_yellow_or_green_no_red_all_hours']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

#Rename columns for clarity
grouped_yellow_df.rename(columns={
    '0-24h': 'yellow_0_24h',
    '24-48h': 'yellow_24_48h',
    '48-72h': 'yellow_48_72h'
}, inplace=True)

block_final = pd.merge(
    block_final,
    grouped_yellow_df,
    on='encounter_block',
    how='left'
)

del grouped_yellow_df

###VENT FREE DAYS
#If dead, 0
#Otherwise uses the last hour of IMV on the hourly data_frame
vent_df = hourly_df[['encounter_block','time_from_vent','hourly_on_vent']]
vent_df['time_from_vent'] = vent_df['time_from_vent'].astype(int)
vent_df['hourly_on_vent'] = vent_df['hourly_on_vent'].astype(bool)
#Keep only values within 28-days and on-vent
vent_df = vent_df[(vent_df['time_from_vent'] <= 28*24) & vent_df['hourly_on_vent'] ]
#Get the MAX hour and merge it into DF
last_vent_df = (
    vent_df.groupby('encounter_block')['time_from_vent']
    .max()
    .reset_index()
)
last_vent_df['time_from_vent'] = last_vent_df['time_from_vent'].astype(int)
last_vent_df.rename(columns={'time_from_vent':'last_hour_on_vent'}, inplace=True)
block_final = pd.merge(
    block_final,
    last_vent_df,
    on='encounter_block',
    how='left'
)
del last_vent_df
del vent_df
#Get an 1 for patients alive at 28-days.
block_final['alive28'] = block_final['date_of_death'].isna() | ((block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() >= (28*24*60*60))
block_final['alive28'] = block_final['alive28'].astype(int)
#Calcute VFD
block_final['vent_free_days'] = block_final['alive28'] * (28 - block_final['last_hour_on_vent']/24)
block_final = block_final.drop(columns=['alive28','last_hour_on_vent'])


del hourly_df


print(f"Block Length: {len(block_final)}")
print(f"Unique Encounter Block: {block_final['encounter_block'].nunique()}")

#Save it to a parquet file
block_final.to_parquet('block_compiled_all_hours.parquet')


/tmp/ipykernel_1417581/2147243220.py:162: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vent_df['time_from_vent'] = vent_df['time_from_vent'].astype(int)
/tmp/ipykernel_1417581/2147243220.py:163: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vent_df['hourly_on_vent'] = vent_df['hourly_on_vent'].astype(bool)


Block Length: 32276
Unique Encounter Block: 32126


In [2]:
#FINALIZE DATA FRAME FOR STATISTICAL ANALYSIS
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np

#Upload final file
block_final = pd.read_parquet('block_compiled_all_hours.parquet')


#Remove Time Outlier (Vent before vitals or vent before admission)
def calc_time_error(col1, col2, error_factor = 1): #Col1 before col2, if not, give error.
    error_list = (col1 - col2).dt.total_seconds()/3600
    error_list = error_list >= error_factor
    return error_list
error_1 = calc_time_error(block_final['block_first_vital_dttm'], block_final['block_vent_start_dttm'])
error_2 = calc_time_error(block_final['admission_dttm'], block_final['block_vent_start_dttm'])
error_3 = error_1 + error_2
block_final = block_final[error_3==0]
print(f"There are {sum(error_3)} encounter blocks where the vent comes more than an hour before admission or before vitals taken. Cohort is now size {len(block_final)}")

#Change relevant DTTM values to hours
block_final['imv_to_discharge_hours'] = (block_final['discharge_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/3600
block_final['imv_to_end_days'] = (block_final['block_vent_end_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/(24*3600)
block_final['adm_to_imv_hours'] = (block_final['block_vent_start_dttm'] - block_final['admission_dttm']).dt.total_seconds()/3600
block_final['imv_to_death_hours'] = (block_final['death_dttm'] - block_final['block_vent_start_dttm']).dt.total_seconds()/3600
block_final['adm_to_discharge_hours'] = (block_final['discharge_dttm'] - block_final['block_first_vital_dttm']).dt.total_seconds()/3600
block_final['yellow_to_discharge_hours'] = block_final['imv_to_discharge_hours'] - block_final['yellow_time_eligibility']

#Add in a dichotomized outcomes variables
block_final['pt_ever'] = block_final['Time_first_PT'].notna()
block_final['pt_post48_IMV'] = block_final['Time_first_PT'].notna() & (block_final['Time_first_PT'] <= 48)
block_final['pt_post48_Yellow'] = block_final['delayed_yellow_PT'].notna() & (block_final['delayed_yellow_PT'] <= 48)
block_final['pt_pre24_IMV'] = block_final['Time_last_PT'].notna() & (block_final['Time_last_PT'] >= -24)
block_final['pt_pre24_or_post48_IMV'] = block_final['pt_pre24_IMV'] | block_final['pt_post48_IMV']
block_final['pt_pre24_or_post48_Yellow'] = block_final['pt_pre24_IMV'] | block_final['pt_post48_Yellow']
block_final['yellow_ever'] = block_final['yellow_time_eligibility'].notna()
block_final['yellow_post48_IMV'] = block_final['yellow_ever'] & (block_final['yellow_time_eligibility'] <= 48)
block_final['extubated_at_pt'] = (block_final['imv_to_end_days']*24) <= block_final['Time_first_PT']
block_final['is_dead'] = block_final['is_dead'].astype(bool)
# Add Hospital mortality: TRUE if Death_dttm < discharge_dttm or (discharge category is hospice or dead) 
block_final["is_dead_hosp"] = (
    (block_final["death_dttm"] <= block_final["discharge_dttm"]) |
    (block_final["discharge_category"].str.lower().isin(["Hospice", "Expired"]))
)
#30-day mortality
block_final['is_dead_30'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (30*24*60*60)
#365-day mortality
block_final['is_dead_365'] = (block_final['date_of_death'] - block_final['block_vent_start_dttm']).dt.total_seconds() <= (365*24*60*60)

#New timme to PT
#Re-defined to include those with a PT order in the 24 hours prior to IMV initiation
block_final['Time_first_PT_new'] = np.where(block_final['pt_pre24_IMV'], block_final['Time_last_PT'], block_final['Time_first_PT'])
block_final['delayed_yellow_PT'] = block_final['Time_first_PT_new'] - block_final['yellow_time_eligibility']
block_final['delayed_yellow_PT_zeroed'] = block_final['delayed_yellow_PT'].apply(lambda x: max(x, 0))

#remove duplicated encounter blocks (for stitched encounters) and save
block_final = block_final.drop_duplicates(subset=['encounter_block'], keep='first')
block_final = block_final.drop(columns=['hospitalization_id'])
block_final.to_parquet('block_with_data_all_hours.parquet')

There are 197 encounter blocks where the vent comes more than an hour before admission or before vitals taken. Cohort is now size 32079


In [3]:
##CLUSTERING OF CATEGORICAL DATA & ORDERING COLUMNS

import pandas as pd
import numpy as np

df = pd.read_parquet('block_with_data_all_hours.parquet')

#Individualy and manually cluster all the categorical variables we want to cluster
print("LANGUAGE PRE:")
print(df['language'].value_counts(dropna=False))
keep = {"English","Spanish"}
set_mask = df['language'].notna() & (~df['language'].eq("None"))
df["language"] = np.where(df["language"].isin(keep), df["language"], "Other")
df["language"] = np.where(set_mask, df['language'], None)
print("LANGUAGE POST:")
print(df['language'].value_counts(dropna=False)) #Print results

print("RACE PRE:")
print(df['race_category'].value_counts(dropna=False))
keep = {"White", "Black or African American"}
set_mask = df['race_category'].notna() & (~df['race_category'].eq("Unknown"))
df["race_category"] = np.where(df["race_category"].isin(keep), df["race_category"], "Other")
df["race_category"] = np.where(set_mask, df['race_category'], None)
print("RACE POST:")
print(df['race_category'].value_counts(dropna=False)) #Print results

print("INSURANCE PRE:")
print(df['insurance'].value_counts(dropna=False))
keep = {"Medicare", "Medicaid","Private"}
set_mask = df['insurance'].notna() & (~df['insurance'].eq("Unknown"))
df["insurance"] = np.where(df["insurance"].isin(keep), df["insurance"], "None/Other")
df["insurance"] = np.where(set_mask, df['insurance'], None)
print("INSURANCE POST:")
print(df['insurance'].value_counts(dropna=False)) #Print results

#This just converts "Unknown" to None for better missingness tracking.
set_mask = df['ethnicity_category'].notna() & (~df['ethnicity_category'].eq("Unknown"))
df["ethnicity_category"] = np.where(set_mask, df['ethnicity_category'], None)

print("ICU TYPE PRE:")
print(df['ICU_type'].value_counts(dropna=False))
mapping = {
    "Medical Intensive Care Unit (MICU)": "Medical ICU",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)": "Medical ICU",
    "Intensive Care Unit (ICU)": "Medical ICU", 
    "Surgical Intensive Care Unit (SICU)": "Surgical ICU",
    "Coronary Care Unit (CCU)": "Cardiac ICU",
    "Cardiac Vascular Intensive Care Unit (CVICU)": "Cardiac ICU",
    "Trauma SICU (TSICU)": "Other",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)": "Other"
}
df['ICU_type'] = df['ICU_type'].map(mapping)
df['ICU_type'] = np.where(df['ICU_type'].notna(), df['ICU_type'], None)
print("ICU TYPE POST:")
print(df['ICU_type'].value_counts(dropna=False))

print("DISCHARGE PRE:")
print(df['discharge_category'].value_counts(dropna=False))
mapping = {
    "Home": "Home",
    "Against Medical Advice (AMA)": "Home",
    "Assisted Living": "Home",
    "Hospice": "Hospice", "Expired": "Expired",
    "Skilled Nursing Facility (SNF)": "Rehabilitation",
    "Acute Inpatient Rehab Facility": "Rehabilitation",
    "Psychiatric Hospital": "Other",
    "Acute Care Hospital": "Other",
    "Long Term Care Hospital (LTACH)": "Other",
    "Other": "Other", "Missing": "Missing"
}
df['discharge_category'] = df['discharge_category'].map(mapping)
print("DISCHARGE POST:")
print(df['discharge_category'].value_counts(dropna=False))


print("ADMISSION SOURCE PRE:")
print(df['admission_source'].value_counts(dropna=False))
##THIS IS A PLACEHOLDER, WE NEED TO UPDATE MIMIC to CLIF MAPPINGS TO GET ADMISSION_TYPE_CATEGORY instead.
mapping = {
    "EW EMER.": "Emergency/Urgent",
    "URGENT": "Emergency/Urgent",
    "EU OBSERVATION": "Observation",
    "OBSERVATION ADMIT": "Observation",
    "DIRECT OBSERVATION": "Observation",
    "SURGICAL SAME DAY ADMISSION": "Surgery",
    "DIRECT EMER.": "Direct",
    "ELECTIVE": "Direct",
    "AMBULATORY OBSERVATION": "Observation",
}
df['admission_source'] = df['admission_source'].map(mapping)
print("ADMISSION SOURCE POST:")
print(df['admission_source'].value_counts(dropna=False))

#Organize and select the columns we want to keep
column_order = pd.read_csv("/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_statistical_analysis/column_order.csv")
column_order['column'] = column_order['column'].astype(str)
column_list = column_order['column'].tolist()
df = df[column_list]

df.to_parquet('block_with_categorical_clustered_all_hours.parquet')


LANGUAGE PRE:
language
English                   28944
Spanish                    1016
Chinese                     352
Russian                     304
Portuguese                  240
Haitian                     202
None                        190
Kabuverdianu                183
Vietnamese                  100
Other                        90
Italian                      69
Arabic                       55
Modern Greek (1453-)         55
Khmer                        33
Persian                      27
Korean                       27
Polish                       24
Hindi                        23
American Sign Language       19
Thai                         13
French                       12
Amharic                      12
Bengali                       9
Armenian                      8
Somali                        1
Japanese                      1
Name: count, dtype: int64
LANGUAGE POST:
language
English    28944
Other       1859
Spanish     1016
None         190
Name: count, dtype: int64
R

In [4]:
#RUN STATS FOR TABLE 1
import pandas as pd
import scipy.stats as stats
import numpy as np
import os

df = pd.read_parquet('block_with_categorical_clustered_all_hours.parquet')
column_order = pd.read_csv("/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_statistical_analysis/column_order.csv")
my_cols = column_order['column'].tolist()
column_order = column_order.set_index('column')

#Convert outcome to a categorical column
early_col = "pt_pre24_or_post48_Yellow"
df["early_PT"] = np.where(df[early_col], "early_PT", "no_early_PT")
n_total = df["encounter_block"].count()
n_early = df[early_col].sum()
n_not = n_total - n_early

file_path = "table1_all_hours.csv"
if os.path.exists(file_path):
    os.remove(file_path)

with open(file_path, mode="w") as file:
    file.write(f",,Overall,Early PT, No Early PT, P-value, Missing")
    for col in my_cols:
        lab = column_order.loc[col,"column_label"]
        if col == 'encounter_block':
            file.write(f"\nN,,{n_total},{n_early},{n_not},")
        elif df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
            file.write(f"\n{lab}")
            cats = df[col].unique()
            p_df = pd.pivot_table(df[["encounter_block",col,"early_PT"]], index=col, columns="early_PT", values="encounter_block", aggfunc='count')
            chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
            for cc in cats:
                if cc: #Because there is a NAN category
                    cc_all = round(100*p_df.loc[cc,].sum()/df[col].count(),1)
                    cc_early = round(100*p_df.loc[cc,'early_PT'].sum()/p_df['early_PT'].sum(),1)
                    cc_not = round(100*p_df.loc[cc,'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                    file.write(f"\n,{cc},{cc_all}%,{cc_early}%,{cc_not}%")
            file.write(f", {round(p_value,5)}")
        elif df[col].dtype == "bool" or df[col].dtype == "boolean":
            if df[col].sum(skipna=True) > 0:#Pivot table does not work if there are no true values
                sub_df = df[df[col].notna()]
                sub_df['flag'] = np.where(sub_df[col],"TRUE", "FALSE")
                p_df = pd.pivot_table(sub_df[["encounter_block","flag","early_PT"]], index="flag", columns="early_PT", values="encounter_block", aggfunc='count')
                chi2_stat, p_value, dof, expected = stats.chi2_contingency(p_df.to_numpy())
                cc_all = round(100*p_df.loc["TRUE",].sum()/sub_df[col].count(),1)
                cc_early = round(100*p_df.loc["TRUE",'early_PT'].sum()/p_df['early_PT'].sum(),1)
                cc_not = round(100*p_df.loc["TRUE",'no_early_PT'].sum()/p_df['no_early_PT'].sum(),1)
                file.write(f"\n{lab},,{cc_all}%,{cc_early}%,{cc_not}%,{round(p_value,5)}")
            else:
                file.write(f"\n{lab},,0.00%,0.00%,0.00%,N/A")
        elif pd.api.types.is_numeric_dtype(df[col]):
            cc_all = df[col].dropna()
            cc_early = df.loc[df[early_col], col]
            cc_early = cc_early.dropna()
            cc_not = df.loc[~df[early_col], col]
            cc_not = cc_not.dropna()
            rank_stat, p_value = stats.ranksums(cc_early.tolist(), cc_not.tolist())
            file.write(f"\n{lab} (Med & IQR),,{round(cc_all.median(),2)}  ({round(cc_all.quantile(0.25),2)} - {round(cc_all.quantile(0.75),2)})")
            file.write(f",{round(cc_early.median(),2)}  ({round(cc_early.quantile(0.25),2)} - {round(cc_early.quantile(0.75),2)})")
            file.write(f",{round(cc_not.median(),2)}  ({round(cc_not.quantile(0.25),2)} - {round(cc_not.quantile(0.75),2)})")
            file.write(f",{round(p_value,5)}")
        else:
            file.write(f"\n{lab},ERROR,,,,,")
        #Missing data column
        mis_pct = round(100*sum(df[col].isna())/n_total,2)
        file.write(f",{mis_pct}%")
            


/tmp/ipykernel_1417581/1325822769.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['flag'] = np.where(sub_df[col],"TRUE", "FALSE")


In [6]:
df = pd.read_parquet('block_with_data_all_hours.parquet')
df['patient_id'] = df['patient_id'].astype(str)
print(f"Unique patients: {df['patient_id'].nunique()}")

Unique patients: 29296


In [29]:
####NEXT STEPS
"""
- Vent free days
- Renal replacement therapy? 
"""


'\n- Vent free days\n- Renal replacement therapy? \n'